# AML HW1

Heart Disease classification workflow: raw data -> cleaned/processed data -> model comparison.

## Workflow Overview
1. Load and inspect raw data (including missing-value locations).
2. Create binary target and define base vs engineered feature sets.
3. Save processed feature table for reproducibility.
4. Split into train/test sets (stratified).
5. Fit train-only imputation + categorical encoding; train/evaluate Naive Bayes variants.
6. Reuse train-only imputation; train/evaluate Linear Regression, Ridge, and Lasso (with threshold analysis).
7. Visualize confusion matrices, metric comparisons, and ROC/AUC.
8. Run feature ablation (pre vs post engineered features) for LinearRegression and GaussianNB.


In [ ]:
# Imports and shared helpers used throughout the notebook.
# Keeping this logic in one place makes later comparisons easier to trust and review.
from pathlib import Path

import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
)

pd.set_option("display.max_columns", 200)
SEED = 42

CATEGORICAL_CANDIDATES = [
    "sex", "cp", "fbs", "restecg", "exang", "slope", "thal",
    "ca_missing", "thal_missing", "thal_is_abnormal"
]


# compute_classification_metrics: returns the standard classification metrics used throughout the notebook.
def compute_classification_metrics(y_true, y_pred, y_score=None):
    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "cm": confusion_matrix(y_true, y_pred),
        "report": classification_report(y_true, y_pred, digits=3, zero_division=0),
    }
    if y_score is not None:
        result["y_score"] = y_score
    return result


# print_metrics_block: prints one model's results in the same format each time.
def print_metrics_block(model_name, res):
    print(f"\n{model_name}")
    if "var_smoothing" in res:
        print(f"var_smoothing: {res['var_smoothing']}")
    print(f"Accuracy : {res['accuracy']:.4f}")
    print(f"Precision: {res['precision']:.4f}")
    print(f"Recall   : {res['recall']:.4f}")
    print(f"F1-score : {res['f1']:.4f}")
    print(res["report"])


# impute_and_encode_train_test: fits median imputation and one-hot encoding on train, then applies them to test.
def impute_and_encode_train_test(X_train_df, X_test_df):
    """Train-only median imputation + one-hot encoding used across Steps 5/8."""
    imputer = SimpleImputer(strategy="median")
    X_train_imp = pd.DataFrame(
        imputer.fit_transform(X_train_df), columns=X_train_df.columns, index=X_train_df.index
    )
    X_test_imp = pd.DataFrame(
        imputer.transform(X_test_df), columns=X_test_df.columns, index=X_test_df.index
    )

    categorical_cols = [col for col in CATEGORICAL_CANDIDATES if col in X_train_imp.columns]
    numeric_cols = [col for col in X_train_imp.columns if col not in categorical_cols]

    encoder = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
            ("num", "passthrough", numeric_cols),
        ],
        remainder="drop",
    )

    X_train_enc = encoder.fit_transform(X_train_imp)
    X_test_enc = encoder.transform(X_test_imp)
    feature_names = encoder.get_feature_names_out()

    X_train_enc = pd.DataFrame(X_train_enc, columns=feature_names, index=X_train_imp.index)
    X_test_enc = pd.DataFrame(X_test_enc, columns=feature_names, index=X_test_imp.index)

    return X_train_imp, X_test_imp, X_train_enc, X_test_enc


In [ ]:
# Step 1: load the Cleveland subset and inspect obvious data quality issues.
columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
    "exang", "oldpeak", "slope", "ca", "thal", "target"
]

raw_path = Path("..") / "data" / "raw" / "processed.cleveland.data"
if not raw_path.exists():
    raw_path = Path("data") / "raw" / "processed.cleveland.data"

# '?' indicates missing values in this file.
df_raw = pd.read_csv(raw_path, header=None, names=columns, na_values="?")

# Parse missing tokens and coerce all columns to numeric.
for c in df_raw.columns:
    df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce")

print("Raw shape:", df_raw.shape)
print("Raw missing values (count by column):")
print(df_raw.isna().sum())

print("\nRaw data preview:")
display(df_raw.head())

rows_any_missing = df_raw[df_raw.isna().any(axis=1)].index.tolist()
rows_missing_ca = df_raw[df_raw["ca"].isna()].index.tolist()
rows_missing_thal = df_raw[df_raw["thal"].isna()].index.tolist()

print("\nRows with any missing values (0-based index):", rows_any_missing)
print("Rows missing ca (0-based index):", rows_missing_ca)
print("Rows missing thal (0-based index):", rows_missing_thal)

# Show both a normal preview and the specific rows that need cleaning attention.
print("\nRows with missing values (subset table):")
display(df_raw.loc[rows_any_missing])


In [ ]:
# Step 2: define the binary target and separate base vs engineered feature sets.
# Binary target: 0 = no disease, 1 = disease present.
df = df_raw.copy()
df["target_bin"] = (df["target"] > 0).astype(int)

# Missingness can be informative here, so preserve it as explicit indicator columns.
df["ca_missing"] = df["ca"].isna().astype(int)
df["thal_missing"] = df["thal"].isna().astype(int)

# Base feature set used for the pre-engineering comparison.
feature_cols_base = [col for col in df.columns if col not in ["target", "target_bin"]]
df_features_base = df[feature_cols_base].copy()

# Enhanced feature set adds a small number of interpretable derived signals.
df_features_eng = df_features_base.copy()
# maxhr_age_ratio: peak heart rate normalized by age.
df_features_eng["maxhr_age_ratio"] = df_features_eng["thalach"] / df_features_eng["age"]
# exercise_risk: combines exercise-related chest pain with the oldpeak measurement.
df_features_eng["exercise_risk"] = df_features_eng["exang"] * df_features_eng["oldpeak"]
# Keep missing thal as NaN here so the imputer learns from train only later on.
# thal_is_abnormal: indicator that thal test result is not normal (3).
df_features_eng["thal_is_abnormal"] = np.where(
    df_features_eng["thal"].isna(), np.nan, (df_features_eng["thal"] != 3).astype(float)
)

# Keep target column last.
df_processed_base = df_features_base.copy()
df_processed_base["target_bin"] = df["target_bin"].values

df_processed = df_features_eng.copy()
df_processed["target_bin"] = df["target_bin"].values

print("Processed (enhanced) shape:", df_processed.shape)
print("Missing values in enhanced features before split-imputation:")
print(df_processed.isna().sum())
display(df_processed.head())


In [ ]:
# Step 3: save the processed table so the modeling input is reproducible.
# Saving before split/imputation avoids baking train/test information into the artifact.
processed_path = Path("..") / "data" / "processed" / "heart_cleveland_clean.csv"
if not processed_path.parent.exists():
    processed_path = Path("data") / "processed" / "heart_cleveland_clean.csv"

processed_path.parent.mkdir(parents=True, exist_ok=True)
df_processed.to_csv(processed_path, index=False)

print("Saved processed data to:", processed_path)


In [ ]:
# Step 4: create one shared stratified split for all model comparisons.
X = df_processed.drop(columns=["target_bin"])
y = df_processed["target_bin"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("\nTrain class balance:")
print(y_train.value_counts(normalize=True).sort_index())
print("\nTest class balance:")
print(y_test.value_counts(normalize=True).sort_index())


In [ ]:
# Step 5: fit and compare the Naive Bayes variants.
# Fit imputation/encoding on the training set only, then apply the same transforms to test.
X_train_imp, X_test_imp, X_train_enc, X_test_enc = impute_and_encode_train_test(X_train, X_test)

# This confirms that missing values are resolved without leaking test information.
missing_before_train = X_train.isna().sum().sum()
missing_before_test = X_test.isna().sum().sum()
missing_after_train = X_train_imp.isna().sum().sum()
missing_after_test = X_test_imp.isna().sum().sum()

missing_summary = pd.DataFrame({
    "dataset": ["Train", "Test"],
    "before_imputation": [missing_before_train, missing_before_test],
    "after_imputation": [missing_after_train, missing_after_test],
})

print("Missing values before vs after train-only imputation:")
display(missing_summary)

# Show row-level before/after examples so the cleaning step is concrete, not just summarized.
missing_row_idx = df_raw[df_raw.isna().any(axis=1)].index
train_missing_idx = X_train.index.intersection(missing_row_idx)
test_missing_idx = X_test.index.intersection(missing_row_idx)

compare_cols = [col for col in ["ca", "thal", "ca_missing", "thal_missing"] if col in X_train.columns]

def build_before_after_table(X_before, X_after, idx, set_name):
    before = X_before.loc[idx, compare_cols].copy()
    before["set"] = set_name
    before["stage"] = "before"

    after = X_after.loc[idx, compare_cols].copy()
    after["set"] = set_name
    after["stage"] = "after"

    compare = pd.concat([before, after]).reset_index().rename(columns={"index": "row_id"})
    stage_order = pd.CategoricalDtype(categories=["before", "after"], ordered=True)
    compare["stage"] = compare["stage"].astype(stage_order)
    compare = compare.sort_values(["row_id", "stage"], kind="mergesort").set_index("row_id")
    return compare

if len(train_missing_idx) > 0:
    train_compare = build_before_after_table(X_train, X_train_imp, train_missing_idx, "train")
    print("\nTrain rows with original missing values: before vs after imputation")
    display(train_compare)

if len(test_missing_idx) > 0:
    test_compare = build_before_after_table(X_test, X_test_imp, test_missing_idx, "test")
    print("\nTest rows with original missing values: before vs after imputation")
    display(test_compare)

# BernoulliNB expects binary inputs, so threshold encoded features using train medians.
train_medians = X_train_enc.median()
X_train_bin = (X_train_enc > train_medians).astype(int)
X_test_bin = (X_test_enc > train_medians).astype(int)

nb_results = {}
# Include a true no-smoothing baseline so the effect of add-alpha smoothing is visible.
for alpha in [0.0, 0.01, 1.0]:
    nb_model = BernoulliNB(alpha=alpha, force_alpha=True)
    if alpha == 0.0:
        # With alpha=0.0, zero probabilities are expected; suppress the resulting runtime noise only for this baseline.
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message=".*divide by zero encountered in log.*", category=RuntimeWarning)
            warnings.filterwarnings("ignore", message=".*invalid value encountered in matmul.*", category=RuntimeWarning)
            nb_model.fit(X_train_bin, y_train)
            y_pred_nb = nb_model.predict(X_test_bin)
            y_score_nb = nb_model.predict_proba(X_test_bin)[:, 1]
    else:
        nb_model.fit(X_train_bin, y_train)
        y_pred_nb = nb_model.predict(X_test_bin)
        y_score_nb = nb_model.predict_proba(X_test_bin)[:, 1]

    nb_results[f"BernoulliNB(alpha={alpha})"] = compute_classification_metrics(
        y_test, y_pred_nb, y_score=y_score_nb
    )

no_smooth_key = "BernoulliNB(alpha=0.0)"
smooth_keys = [k for k in nb_results if k.startswith("BernoulliNB") and k != no_smooth_key]
best_smooth_key = max(smooth_keys, key=lambda k: nb_results[k]["accuracy"])
no_smooth = nb_results[no_smooth_key]
best_smooth = nb_results[best_smooth_key]
delta_acc = best_smooth["accuracy"] - no_smooth["accuracy"]
tn0, fp0, fn0, tp0 = no_smooth["cm"].ravel()
tn1, fp1, fn1, tp1 = best_smooth["cm"].ravel()
print("\nBernoulli smoothing impact (no smoothing vs smoothing):")
print(f"No smoothing: {no_smooth_key} | accuracy={no_smooth['accuracy']:.4f}")
print(f"Best smoothing: {best_smooth_key} | accuracy={best_smooth['accuracy']:.4f}")
print(f"Accuracy change (smoothed - unsmoothed): {delta_acc:+.4f}")
print(f"FP change: {fp1 - fp0:+d}, FN change: {fn1 - fn0:+d}")

# Tune GaussianNB over a small var_smoothing grid and keep the best setting.
gaussian_candidates = {}
for vs in [1e-12, 1e-11, 1e-10, 1e-9, 1e-8]:
    gnb = GaussianNB(var_smoothing=vs)
    gnb.fit(X_train_enc, y_train)
    y_pred_gnb = gnb.predict(X_test_enc)
    y_score_gnb = gnb.predict_proba(X_test_enc)[:, 1]

    gaussian_candidates[vs] = compute_classification_metrics(
        y_test, y_pred_gnb, y_score=y_score_gnb
    )

best_vs = max(gaussian_candidates, key=lambda v: gaussian_candidates[v]["accuracy"])
nb_results["GaussianNB"] = gaussian_candidates[best_vs]
nb_results["GaussianNB"]["var_smoothing"] = best_vs
print(f"Selected GaussianNB var_smoothing={best_vs}")

for model_name, res in nb_results.items():
    print_metrics_block(model_name, res)


In [ ]:
# Step 6: fit the linear-family models on the same processed split.
# Reuse the same encoded features from Step 5, then standardize for the linear models.
scaler_lr = StandardScaler()
X_train_lr = scaler_lr.fit_transform(X_train_enc)
X_test_lr = scaler_lr.transform(X_test_enc)

lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train)
y_pred_lr_cont = lr_model.predict(X_test_lr)

# Ridge and Lasso are included to show how simple regularization changes behavior.
ridge_model = Ridge(alpha=1.0, random_state=SEED)
ridge_model.fit(X_train_lr, y_train)
y_pred_ridge_cont = ridge_model.predict(X_test_lr)

lasso_model = Lasso(alpha=0.01, random_state=SEED, max_iter=20000)
lasso_model.fit(X_train_lr, y_train)
y_pred_lasso_cont = lasso_model.predict(X_test_lr)

# Sweep a few thresholds to make the precision/recall tradeoff visible.
thresholds = [0.3, 0.5, 0.7]
lr_threshold_results = {}

for thr in thresholds:
    y_pred_lr = (y_pred_lr_cont >= thr).astype(int)
    lr_threshold_results[thr] = compute_classification_metrics(y_test, y_pred_lr)

print("LinearRegression threshold analysis")
for thr, res in lr_threshold_results.items():
    print(f"\nThreshold={thr}")
    print(f"Accuracy : {res['accuracy']:.4f}")
    print(f"Precision: {res['precision']:.4f}")
    print(f"Recall   : {res['recall']:.4f}")
    print(f"F1-score : {res['f1']:.4f}")
    print(res["report"])

# Use 0.5 as the main threshold so comparisons stay aligned with the assignment.
y_pred_lr = (y_pred_lr_cont >= 0.5).astype(int)
lr_accuracy = lr_threshold_results[0.5]["accuracy"]
lr_precision = lr_threshold_results[0.5]["precision"]
lr_recall = lr_threshold_results[0.5]["recall"]
lr_f1 = lr_threshold_results[0.5]["f1"]
lr_cm = lr_threshold_results[0.5]["cm"]

# Keep the same threshold for Ridge and Lasso to make the comparison fair.
y_pred_ridge = (y_pred_ridge_cont >= 0.5).astype(int)
ridge_res = compute_classification_metrics(y_test, y_pred_ridge)
ridge_accuracy = ridge_res["accuracy"]
ridge_precision = ridge_res["precision"]
ridge_recall = ridge_res["recall"]
ridge_f1 = ridge_res["f1"]
ridge_cm = ridge_res["cm"]

print("\nRidge Regression (alpha=1.0, threshold=0.5)")
print(f"Accuracy : {ridge_accuracy:.4f}")
print(f"Precision: {ridge_precision:.4f}")
print(f"Recall   : {ridge_recall:.4f}")
print(f"F1-score : {ridge_f1:.4f}")
print(ridge_res["report"])

y_pred_lasso = (y_pred_lasso_cont >= 0.5).astype(int)
lasso_res = compute_classification_metrics(y_test, y_pred_lasso)
lasso_accuracy = lasso_res["accuracy"]
lasso_precision = lasso_res["precision"]
lasso_recall = lasso_res["recall"]
lasso_f1 = lasso_res["f1"]
lasso_cm = lasso_res["cm"]

print("\nLasso Regression (alpha=0.01, threshold=0.5)")
print(f"Accuracy : {lasso_accuracy:.4f}")
print(f"Precision: {lasso_precision:.4f}")
print(f"Recall   : {lasso_recall:.4f}")
print(f"F1-score : {lasso_f1:.4f}")
print(lasso_res["report"])


In [ ]:
# Step 7: summarize model behavior visually and numerically.
# A) Confusion matrices for the main models discussed in the report.
best_bernoulli = max(
    [k for k in nb_results.keys() if k.startswith("BernoulliNB")],
    key=lambda k: nb_results[k]["accuracy"]
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, name, cmap in zip(
    axes,
    [best_bernoulli, "GaussianNB", "LinearRegression(thr=0.5)"],
    ["Blues", "Purples", "Greens"]
):
    cm = nb_results[name]["cm"] if name in nb_results else lr_cm
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, cbar=False, ax=ax, linewidths=0.5)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

# B) Side-by-side metric comparison across all fitted models.
comparison_rows = []
for model_name, res in nb_results.items():
    comparison_rows.append({"model": model_name, "accuracy": res["accuracy"], "precision": res["precision"], "recall": res["recall"], "f1": res["f1"]})
comparison_rows.append({"model": "LinearRegression(thr=0.5)", "accuracy": lr_accuracy, "precision": lr_precision, "recall": lr_recall, "f1": lr_f1})
comparison_rows.append({"model": "Ridge(alpha=1.0)", "accuracy": ridge_accuracy, "precision": ridge_precision, "recall": ridge_recall, "f1": ridge_f1})
comparison_rows.append({"model": "Lasso(alpha=0.01)", "accuracy": lasso_accuracy, "precision": lasso_precision, "recall": lasso_recall, "f1": lasso_f1})

comparison = pd.DataFrame(comparison_rows).sort_values(by="accuracy", ascending=False)
display(comparison)

long_cmp = comparison.melt(id_vars="model", var_name="metric", value_name="value")
plt.figure(figsize=(11.5, 4.8))
sns.barplot(data=long_cmp, x="metric", y="value", hue="model")
plt.ylim(0, 1.0)
plt.title("Model Metric Comparison")
plt.ylabel("Score")
plt.xlabel("")
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# C) Threshold tradeoff for LinearRegression.
thr_df = pd.DataFrame([
    {"threshold": t, "accuracy": r["accuracy"], "precision": r["precision"], "recall": r["recall"], "f1": r["f1"]}
    for t, r in lr_threshold_results.items()
]).sort_values("threshold")

display(thr_df)

plt.figure(figsize=(8, 4.5))
for metric in ["accuracy", "precision", "recall", "f1"]:
    plt.plot(thr_df["threshold"], thr_df[metric], marker="o", label=metric)
plt.ylim(0, 1.0)
plt.xticks(thr_df["threshold"])
plt.title("Linear Regression: Metric vs Classification Threshold")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend()
plt.tight_layout()
plt.show()

# D) ROC curves and AUC values (bonus analysis).
plt.figure(figsize=(7.2, 5.2))
for model_name in [best_bernoulli, "GaussianNB"]:
    fpr, tpr, _ = roc_curve(y_test, nb_results[model_name]["y_score"])
    model_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC={model_auc:.3f})")

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_pred_lr_cont)
auc_lr = auc(fpr_lr, tpr_lr)
plt.plot(fpr_lr, tpr_lr, linewidth=2, label=f"LinearRegression (AUC={auc_lr:.3f})")

fpr_ridge, tpr_ridge, _ = roc_curve(y_test, y_pred_ridge_cont)
auc_ridge = auc(fpr_ridge, tpr_ridge)
plt.plot(fpr_ridge, tpr_ridge, linewidth=2, label=f"Ridge (AUC={auc_ridge:.3f})")

fpr_lasso, tpr_lasso, _ = roc_curve(y_test, y_pred_lasso_cont)
auc_lasso = auc(fpr_lasso, tpr_lasso)
plt.plot(fpr_lasso, tpr_lasso, linewidth=2, label=f"Lasso (AUC={auc_lasso:.3f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.6)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
# Step 8: isolate the effect of engineered features with a controlled ablation.
# Everything stays the same between branches except the presence of engineered features.

y_cmp = df["target_bin"]
X_base = df_processed_base.drop(columns=["target_bin"])
X_eng = df_processed.drop(columns=["target_bin"])

# Reuse the same split indices so any difference can be attributed to the features.
idx_train, idx_test = train_test_split(
    df.index, test_size=0.2, random_state=SEED, stratify=y_cmp
)

yb_train, yb_test = y_cmp.loc[idx_train], y_cmp.loc[idx_test]
ye_train, ye_test = y_cmp.loc[idx_train], y_cmp.loc[idx_test]
Xb_train, Xb_test = X_base.loc[idx_train], X_base.loc[idx_test]
Xe_train, Xe_test = X_eng.loc[idx_train], X_eng.loc[idx_test]

_, _, Xb_train_enc, Xb_test_enc = impute_and_encode_train_test(Xb_train, Xb_test)
_, _, Xe_train_enc, Xe_test_enc = impute_and_encode_train_test(Xe_train, Xe_test)

# LinearRegression ablation uses the same encoding and scaling policy as the main run.
sc_base = StandardScaler()
Xb_train_s = sc_base.fit_transform(Xb_train_enc)
Xb_test_s = sc_base.transform(Xb_test_enc)
lr_base = LinearRegression().fit(Xb_train_s, yb_train)
yb_pred = (lr_base.predict(Xb_test_s) >= 0.5).astype(int)
acc_lr_base = accuracy_score(yb_test, yb_pred)

sc_eng = StandardScaler()
Xe_train_s = sc_eng.fit_transform(Xe_train_enc)
Xe_test_s = sc_eng.transform(Xe_test_enc)
lr_eng = LinearRegression().fit(Xe_train_s, ye_train)
ye_pred = (lr_eng.predict(Xe_test_s) >= 0.5).astype(int)
acc_lr_eng = accuracy_score(ye_test, ye_pred)

# GaussianNB ablation reuses the tuned Step 5 setting for a clean apples-to-apples check.
vs_ablation = nb_results.get("GaussianNB", {}).get("var_smoothing", 1e-9)
gnb_base = GaussianNB(var_smoothing=vs_ablation).fit(Xb_train_enc, yb_train)
acc_gnb_base = accuracy_score(yb_test, gnb_base.predict(Xb_test_enc))

gnb_eng = GaussianNB(var_smoothing=vs_ablation).fit(Xe_train_enc, ye_train)
acc_gnb_eng = accuracy_score(ye_test, gnb_eng.predict(Xe_test_enc))

feature_gain = pd.DataFrame([
    {"model": "LinearRegression", "pre_features_acc": acc_lr_base, "post_features_acc": acc_lr_eng},
    {"model": "GaussianNB", "pre_features_acc": acc_gnb_base, "post_features_acc": acc_gnb_eng},
])
feature_gain["delta"] = feature_gain["post_features_acc"] - feature_gain["pre_features_acc"]

print("Target_bin accuracy before vs after engineered features:")
display(feature_gain)

# Plot pre/post accuracy to make the feature effect easy to read quickly.
plot_df = feature_gain.melt(
    id_vars="model",
    value_vars=["pre_features_acc", "post_features_acc"],
    var_name="stage",
    value_name="accuracy",
)
plot_df["stage"] = plot_df["stage"].map({
    "pre_features_acc": "Pre Features",
    "post_features_acc": "Post Features",
})

plt.figure(figsize=(8.5, 4.8))
sns.barplot(data=plot_df, x="model", y="accuracy", hue="stage")
plt.ylim(0, 1.0)
plt.title("Feature Impact: Accuracy Before vs After")
plt.xlabel("")
plt.ylabel("Accuracy")
plt.legend(title="")
plt.tight_layout()
plt.show()

if (feature_gain["delta"] > 0).any():
    print("Result: engineered features improved accuracy for at least one model.")
elif (feature_gain["delta"] == 0).all():
    print("Result: engineered features did not change accuracy on this split.")
else:
    print("Result: engineered features reduced accuracy on this split; keep only features that generalize.")
